> ### **Building with Sarvam**
>
> **An open teaching kit for the Sarvam AI stack.**
>
> Notebook maintained by **Dr. Bhaveshkumar C. Dharmani** — Founder & AI Mentor, AIVidhya4Sarvam.
PhD (ICT), DA-IICT Gandhinagar · https://www.aividhya.in/ · https://www.aividhya4sarvam.in/ · bhavesh@aividhya.in · https://www.linkedin.com/in/bhaveshdharmani/
>
> Drafted with AI assistance and stress-tested cell by cell in live workshop sessions. Runs against your own Sarvam API key from dashboard.sarvam.ai, with a live ₹ cost meter after every call.
>
> Apache 2.0 · Issues and PRs welcome at github.com/dharmanibc/building-with-sarvam.

<div style="background:#12172E;color:#fff;padding:20px 24px;border-radius:8px">
<div style="color:#FF8A3D;font-size:12px;letter-spacing:2px;font-weight:700">LAB 04 · THE LANGUAGE LAYER</div>
<div style="font-size:26px;font-weight:700;margin-top:6px">Translate, transliterate, detect — and one silent failure</div>
<div style="color:#FFB37A;font-size:14px;margin-top:8px">Mayura vs Sarvam-Translate · registers · script conversion · auto-routing pipeline</div>
</div>

**Time:** 40 min &nbsp;·&nbsp; **Est. cost:** ≈ ₹3 &nbsp;·&nbsp; **Prereq:** Lab 00

In [1]:
# ── Standard lab header. Run this first in every notebook. ─────────────────
import os, sys, json, time, math, wave, io
from pathlib import Path
# pip install sarvamai python-dotenv

from dotenv import load_dotenv, find_dotenv
# Finds your key without hardcoding anyone's filesystem. Tried in order:
#   1. SARVAM_API_KEY already set in the environment
#   2. the file named by SARVAM_ENV_FILE, if you set that variable
#   3. a .env beside this notebook, or in any parent folder
load_dotenv(os.environ.get("SARVAM_ENV_FILE") or find_dotenv(usecwd=True))

API_KEY = os.environ.get("SARVAM_API_KEY")
assert API_KEY, (
    "SARVAM_API_KEY not found.\\n"
    "Create a .env next to this notebook containing:  SARVAM_API_KEY=sk_...\\n"
    "or point SARVAM_ENV_FILE at an existing env file.\\n"
    "Free key + Rs 1000 credit: https://indus.sarvam.ai/"
)

from sarvamai import SarvamAI

client = SarvamAI(api_subscription_key=API_KEY)
DATA = Path("./data"); DATA.mkdir(exist_ok=True)
OUT  = Path("./out");  OUT.mkdir(exist_ok=True)
print("SDK ready ·", sys.version.split()[0])

SDK ready · 3.13.9


In [2]:
# ── The ₹ meter, imported ─────────────────────────────────────────────────
# Lab 00 writes cost_meter.py next to these notebooks. If this import fails,
# run Lab 00 once — it is the only lab that defines the meter.
try:
    from cost_meter import CostMeter, RATES, FREE_CREDIT
except ImportError:
    raise ImportError(
        "cost_meter.py not found.\n"
        "Run 00_Setup_and_the_Cost_Meter.ipynb once — its last section writes "
        "cost_meter.py into this folder, and every other lab imports it from there."
    )

cost = CostMeter()
print(f"cost meter armed · rates dated Aug 2026 · ₹{FREE_CREDIT:.0f} free credit")


cost meter armed · rates dated Aug 2026 · ₹1000 free credit


## 1 · Two translation models. Choose on coverage first, quality second.

| | Mayura v1 | Sarvam-Translate v1 |
|---|---|---|
| Languages | 11 | **23** |
| Weights | closed | **open (Apache 2.0)** |
| Price | ₹20 / 10K chars | ₹20 / 10K chars |
| `output_script` | works | **silently ignored** |

In [3]:
SRC = "Your insurance policy will lapse on 30 September unless the premium is paid."

for model in ["mayura:v1", "sarvam-translate:v1"]:
    r = client.text.translate(
        input=SRC, source_language_code="en-IN",
        target_language_code="hi-IN", model=model)
    cost.text(len(SRC), "translate")
    print(f"{model:<22} {r.translated_text}")

mayura:v1              यदि प्रीमियम का भुगतान नहीं किया जाता है तो आपका बीमा पॉलिसी 30 सितंबर को समाप्त हो जाएगा।
sarvam-translate:v1    यदि प्रीमियम का भुगतान नहीं किया गया तो आपकी बीमा पॉलिसी 30 सितंबर को समाप्त हो जाएगी।


### The registers — same meaning, very different product

In [4]:
for mode in ["classic-colloquial",  "modern-colloquial",  "code-mixed",  "formal"]:     #["colloquial", "modern", "classical", "formal"]:
    r = client.text.translate(
        input=SRC, source_language_code="en-IN", target_language_code="hi-IN",
        model="mayura:v1", mode=mode)
    cost.text(len(SRC), "translate")
    print(f"{mode:<11} │ {r.translated_text}")

classic-colloquial │ अगर आप premium नहीं भरते हैं, तो 30 September को आपकी insurance policy ख़त्म हो जाएगी।
modern-colloquial │ अगर आप premium pay नहीं करते हैं, तो 30 September को आपकी insurance policy lapse हो जाएगी।
code-mixed  │ अगर आप premium pay नहीं करते हैं, तो 30 September को आपकी insurance policy lapse हो जाएगी।
formal      │ यदि प्रीमियम का भुगतान नहीं किया जाता है तो आपका बीमा पॉलिसी 30 सितंबर को समाप्त हो जाएगा।


> A government notice rendered in `colloquial` reads like a WhatsApp forward. A support
> chatbot in `classical` sounds absurd. Same API call, completely different product.

---
## 2 · ⚠️ `output_script` on `sarvam-translate:v1` — fixed, but still worth knowing

Until recently, `output_script` was **silently ignored** on `sarvam-translate:v1` —
HTTP 200, wrong script, no exception, no warning. **Sarvam has since fixed this**:
the identical request now raises an explicit `400`. That's a genuine improvement —
but the underlying fact is unchanged: **`output_script` still isn't supported on
this model.** You just find out immediately instead of discovering it in production.

In [5]:
# ⚠️ Trigger it on purpose — this used to silently return roman text (it didn't).
try:
    client.text.translate(
        input=SRC, source_language_code="en-IN", target_language_code="hi-IN",
        model="sarvam-translate:v1",
        output_script="roman",          # <- now rejected outright, not ignored
    )
except Exception as e:
    print(f"{type(e).__name__}: {e}")
    print("\n↑ output_script is not supported on sarvam-translate:v1 at all.")

# The plain call (no output_script) still works — this is what the next
# cell's transliterate fallback post-processes.
r = client.text.translate(
    input=SRC, source_language_code="en-IN", target_language_code="hi-IN",
    model="sarvam-translate:v1")
cost.text(len(SRC), "translate")
print("\nsarvam-translate (no output_script):", r.translated_text)

BadRequestError: headers: {'date': 'Fri, 28 Aug 2026 05:13:12 GMT', 'content-type': 'application/json', 'content-length': '260', 'connection': 'keep-alive', 'server': 'uvicorn', 'x-request-id': '20260828_c91ea74f-48af-4b2e-9400-c297b0eeb5cd', 'access-control-allow-origin': '*', 'strict-transport-security': 'max-age=31536000; includeSubDomains', 'x-frame-options': 'DENY', 'x-content-type-options': 'nosniff', 'referrer-policy': 'strict-origin-when-cross-origin', 'permissions-policy': 'geolocation=(), camera=(), microphone=(), payment=()', 'content-security-policy': "default-src 'none'; frame-ancestors 'none'", 'cache-control': 'no-store, no-cache, must-revalidate', 'pragma': 'no-cache', 'expires': '0', 'x-xss-protection': '0'}, status_code: 400, body: {'error': {'message': 'Transliteration is not supported in sarvam-translate:v1. Please use mayura:v1 for transliteration support or remove the output_script parameter.', 'code': 'invalid_request_error', 'request_id': '20260828_c91ea74f-48af

In [6]:
# ✅ Mayura honours it
r2 = client.text.translate(
    input=SRC, source_language_code="en-IN", target_language_code="hi-IN",
    model="mayura:v1", output_script="roman")
cost.text(len(SRC), "translate")
print("mayura + roman            :", r2.translated_text)

# ✅ Or post-process with transliterate
r3 = client.text.transliterate(
    input=r.translated_text, source_language_code="hi-IN",
    target_language_code="hi-IN", spoken_form=True)
cost.text(len(r.translated_text), "transliterate")
print("sarvam-translate + translit:", r3.transliterated_text)

mayura + roman            : Yadi premium ka bhugtaan nahi kiya jata hai toh aapka Bima Policy 30 September ko khatam ho jaayega.
sarvam-translate + translit: यदि प्रीमियम का भुगतान नहीं किया गया तो आपकी बीमा पॉलिसी थर्टीथ सितम्बर को समाप्त हो जाएगी।


**Why this class of bug is the dangerous one.** An exception costs you five minutes.
A 200 response with subtly wrong output ships to production and gets discovered by a
customer. This is exactly what Agent Skills exist to prevent —
`npx skills add sarvamai/skills`.

---
## 3 · Transliterate — the quiet workhorse

In [7]:
NAMES = ["राजेश कुमार शर्मा", "तिरुवनंतपुरम", "भुवनेश्वर", "₹1,20,000"]

for n in NAMES:
    r = client.text.transliterate(
        input=n, source_language_code="hi-IN",
        target_language_code="en-IN", spoken_form=True)
    cost.text(len(n), "transliterate")
    print(f"{n:<22} → {r.transliterated_text}")

राजेश कुमार शर्मा      → Rajesh Kumar Sharma
तिरुवनंतपुरम           → Thiruvananthapuram
भुवनेश्वर              → Bhuvneshwar
₹1,20,000              → ₹1,20,000


**Where it earns its keep:** cross-script search, KYC name matching, Roman-script UIs,
databases that cannot store Indic Unicode, and legacy system integration. Unglamorous,
constantly needed.

`spoken_form=True` also expands numerals and currency the way a person would say them —
which is exactly what you want before sending text to TTS.

---
## 4 · Language ID as a router

In [8]:
INBOX = [
    "मेरा payment fail हो गया",
    "எனது கட்டணம் தோல்வியடைந்தது",
    "my payment failed",
    "mera payment fail ho gaya",
    "আমার পেমেন্ট ব্যর্থ হয়েছে",
]

for msg in INBOX:
    r = client.text.identify_language(input=msg)
    cost.text(len(msg), "lid")
    print(f"{r.language_code:>8} / {getattr(r,'script_code','—'):<10} {msg}")

   hi-IN / Deva       मेरा payment fail हो गया
   ta-IN / Taml       எனது கட்டணம் தோல்வியடைந்தது
   en-IN / Latn       my payment failed
   pa-IN / Latn       mera payment fail ho gaya
   bn-IN / Beng       আমার পেমেন্ট ব্যর্থ হয়েছে


---
## 5 · Put it together — the auto-routing pipeline

Detect → translate to English → run your (English) business logic → translate back.
This is the single most reusable pattern in Indic product work.

In [9]:
def handle(message: str, meter=cost):
    # 1. What language is this?
    lid = client.text.identify_language(input=message); meter.text(len(message), "lid")
    lang = lid.language_code or "en-IN"

    # 2. Normalise to English so your logic stays monolingual
    if lang != "en-IN":
        t = client.text.translate(input=message, source_language_code=lang,
                                  target_language_code="en-IN", model="mayura:v1")
        meter.text(len(message), "translate")
        english = t.translated_text
    else:
        english = message

    # 3. Your business logic — one language, one set of prompts, one set of tests
    reply_en = business_logic(english)

    # 4. Answer in the language they wrote in
    if lang != "en-IN":
        t = client.text.translate(input=reply_en, source_language_code="en-IN",
                                  target_language_code=lang, model="mayura:v1",
                                  mode="formal")
        meter.text(len(reply_en), "translate")
        return lang, t.translated_text
    return lang, reply_en


def business_logic(text_en: str) -> str:
    r = client.chat.completions(
        model="sarvam-105b", max_tokens=800, reasoning_effort=None,
        messages=[
            {"role": "system", "content": "You are a concise bank support agent. Two sentences maximum."},
            {"role": "user", "content": text_en},
        ])
    cost.llm(r.usage.prompt_tokens, r.usage.completion_tokens)
    return r.choices[0].message.content


for msg in INBOX[:3]:
    lang, reply = handle(msg)
    print(f"\n[{lang}] {msg}\n     → {reply}")


[hi-IN] मेरा payment fail हो गया
     → कृपया अपना लेनदेन आईडी और भुगतान की तारीख प्रदान करें ताकि मैं जाँच कर सकूँ। आपको अपने खाते की गतिविधि या भुगतान की पुष्टि ईमेल में ट्रांजेक्शन आईडी मिल जाएगा।

[ta-IN] எனது கட்டணம் தோல்வியடைந்தது
     → மன்னித்துவிடுங்கள். transaction reference number அல்லது தேதி ஒன்றைத் தயவுசெய்து கூறுங்கள், அப்போதுதான் நான் உங்களுக்காக இதைப் பார்க்க முடியும்.

[en-IN] my payment failed
     → 
I'm sorry your payment failed. Could you please share the transaction ID so I can check the exact reason and help resolve it?


> **The architectural question this raises.** The implementation here is a **design choice, not a technical requirement**. You just paid for two translation calls to
> avoid writing prompts in 11 languages. The alternative is prompting Sarvam-105B
> directly in Hindi / whatever the input language — no translation cost, but now your prompts, tests and evals
> multiply by the number of languages you support. There is no universally right
> answer; there is only the one you costed. 
> 1. **Monolingual business logic** - is one prompt, one system message, one set of few-shot examples/evals. Prompt directly in 23 languages and you now maintain (and regression-test) 23 separate prompt behaviors, or one prompt whose quality varies unevenly across languages you can't all personally read.
> 2. **Tooling/guardrails often assume English** — if business_logic also calls external tools, RAG over English docs, regex/entity extraction, moderation filters, etc., those usually need English input regardless of what the LLM itself can handle.
> 3. **Evaluation is easier in one language** — a human reviewer (or an LLM-judge) grading "did the bot answer correctly" is far cheaper to build for English-only outputs than for 11+ languages, especially early in a project.
>    
>   On the contrary, the disadvantage of this unique language for business logic are: Extra cost + latency, Translation loses nuance, Sarvam-105B degrades this argument's premium - the whole reason to translate-first historically was that reasoning models were English-first, which is not applicable for Sarvam - 105b

In [10]:
cost.report()
print(f"\nPer message handled: ₹{cost.report.__self__.items and sum(i['inr'] for i in cost.items)/max(len(INBOX[:3]),1):.4f}")

translate    ₹   0.1520  76 chars
translate    ₹   0.1520  76 chars
translate    ₹   0.1520  76 chars
translate    ₹   0.1520  76 chars
translate    ₹   0.1520  76 chars
translate    ₹   0.1520  76 chars
translate    ₹   0.1520  76 chars
translate    ₹   0.1520  76 chars
transliterate ₹   0.1720  86 chars
transliterate ₹   0.0340  17 chars
transliterate ₹   0.0240  12 chars
transliterate ₹   0.0180  9 chars
transliterate ₹   0.0180  9 chars
lid          ₹   0.0084  24 chars
lid          ₹   0.0095  27 chars
lid          ₹   0.0059  17 chars
lid          ₹   0.0087  25 chars
lid          ₹   0.0091  26 chars
lid          ₹   0.0084  24 chars
translate    ₹   0.0480  24 chars
LLM          ₹   0.0033  34 in / 31 out
translate    ₹   0.3320  166 chars
lid          ₹   0.0095  27 chars
translate    ₹   0.0540  27 chars
LLM          ₹   0.0031  33 in / 29 out
translate    ₹   0.2420  121 chars
lid          ₹   0.0059  17 chars
LLM          ₹   0.0031  33 in / 29 out
TOTAL        ₹   2.2329
 

---
## ✅ Checkpoint

- [ ] You compared Mayura and Sarvam-Translate on the same input
- [ ] You heard all four registers and can say which suits your product
- [ ] You triggered the `output_script` silent failure and fixed it two ways
- [ ] The auto-routing pipeline answers in the language it was asked in

## 🧪 Try this

1. Run the router on a **romanised** Hindi message. Does LID call it `hi-IN`?
2. Cost the two architectures for 100,000 messages/month: translate-to-English vs
   native-language prompting. Which wins, and at what volume does it flip?
3. Translate a legal paragraph in all four registers and show them to a native speaker.
   Which would you actually send to a customer?